# Análisis comparativo de ROIs (12 / 18 / 39 / 116)

Interfaz de auditoría y comunicación del análisis aprobado en `analysis_plan.md`
(plan estadístico 5.6, congelado). Este notebook **no reimplementa** ninguna
fórmula estadística: importa y llama las funciones públicas de
`scripts/build_analysis_dataset.py` y `scripts/run_statistical_analysis.py`,
exactamente como se invocarían desde la línea de comandos.

No modifica nada bajo `results/`, `src/` ni el código experimental. No
reentrena BrainNetCNN. Reutiliza las 16 corridas ya existentes.

Ejecutar de arriba abajo ("Restart and run all") en CPU. La fase de bootstrap
(10.000 remuestreos x 4 sitios) toma del orden de 15 a 20 minutos; ver la
sección 3 de `analysis/roi_comparison/README.md`.

**Estado de preinscripción (plan 5.6, secciones 1-2 y 14):** este análisis
no es una preinscripción prospectiva ciega a los resultados. El plan se
cerró después de una revisión de factibilidad en la que ya eran visibles las
diferencias medias y varianzas por sitio entre 12 y 116 ROIs. Los resultados
se presentan como estimación con apoyo exploratorio, nunca como confirmación
definitiva.

In [ ]:
import sys
import json
import hashlib
from pathlib import Path

import pandas as pd
from IPython.display import display, Image, Markdown

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "analysis" / "roi_comparison").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("No se encontró la raíz del repositorio (buscando analysis/roi_comparison/)")
    REPO_ROOT = REPO_ROOT.parent

ANALYSIS_DIR = REPO_ROOT / "analysis" / "roi_comparison"
SCRIPTS_DIR = ANALYSIS_DIR / "scripts"
CONFIG_PATH = ANALYSIS_DIR / "config" / "analysis_config.json"
MANIFEST_PATH = ANALYSIS_DIR / "config" / "run_manifest.csv"
OUTPUT_DIR = ANALYSIS_DIR / "outputs"

sys.path.insert(0, str(SCRIPTS_DIR))
print("repo root:", REPO_ROOT)

## 1. Identidad del análisis: versión del plan, configuración, manifiesto, hashes

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

analysis_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
run_manifest = pd.read_csv(MANIFEST_PATH)
plan_path = ANALYSIS_DIR / "analysis_plan.md"

print("plan_version (config):", analysis_config["plan_version"])
print("analysis_plan.md sha256:", sha256_file(plan_path) if plan_path.exists() else "(no presente todavía)")
print("analysis_config.json sha256:", sha256_file(CONFIG_PATH))
print("run_manifest.csv sha256:", sha256_file(MANIFEST_PATH))
print("noninferiority_margin:", analysis_config["noninferiority_margin"])
print("noninferiority_margin_rationale:", analysis_config["noninferiority_margin_rationale"])
print()
display(run_manifest[run_manifest["include"]][["site", "roi_set", "run_id"]])

## 2. Fase 1 — construcción del dataset analítico y auditoría de comparabilidad

Llama a `build_analysis_dataset.main()` exactamente como la interfaz de línea
de comandos documentada. La auditoría se muestra **antes** de cualquier
resultado científico.

In [ ]:
import build_analysis_dataset as bad

sys.argv = [
    "build_analysis_dataset.py",
    "--repo-root", str(REPO_ROOT),
    "--config", str(CONFIG_PATH),
    "--manifest", str(MANIFEST_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--overwrite",
]
build_exit_code = bad.main()
assert build_exit_code == 0, "La construcción del dataset falló; ver mensajes anteriores."
print("build_analysis_dataset.main() -> OK")

In [ ]:
comparability_audit = pd.read_csv(OUTPUT_DIR / "tables" / "comparability_audit.csv")
assert len(comparability_audit) == 16
assert (comparability_audit["status"] == "PASS").all(), "Hay corridas en FAIL; revisar diagnostic."
assert (comparability_audit["reconciliation_status"] == "PASS").all()
display(comparability_audit[["site", "roi_set", "run_id", "status", "reconciliation_status",
                              "recalculated_auc", "published_auc"]])

## 3. Fase 2 — análisis estadístico (bootstrap, contrastes, errores, figuras)

**Nota de rendimiento:** 10.000 remuestreos x 4 sitios toma del orden de 15 a
20 minutos en CPU con `scikit-learn` (ver §3/§16 de las instrucciones de
implementación). Ese costo es esperado y no justifica optimizar,
paralelizar ni sustituir `sklearn.metrics.roc_auc_score`.

In [ ]:
import run_statistical_analysis as rsa

sys.argv = [
    "run_statistical_analysis.py",
    "--repo-root", str(REPO_ROOT),
    "--config", str(CONFIG_PATH),
    "--manifest", str(MANIFEST_PATH),
    "--input-dir", str(OUTPUT_DIR / "data"),
    "--output-dir", str(OUTPUT_DIR),
    "--overwrite",
]
stats_exit_code = rsa.main()
assert stats_exit_code == 0, "El análisis estadístico falló; ver mensajes anteriores."
print("run_statistical_analysis.main() -> OK")

## 4. Tablas

In [ ]:
descriptive_performance = pd.read_csv(OUTPUT_DIR / "tables" / "descriptive_performance.csv")
primary_12_vs_116 = pd.read_csv(OUTPUT_DIR / "tables" / "primary_12_vs_116.csv")
precision_diagnostics = pd.read_csv(OUTPUT_DIR / "tables" / "precision_diagnostics.csv")
secondary_pairwise = pd.read_csv(OUTPUT_DIR / "tables" / "secondary_pairwise_comparisons.csv")
secondary_intervals = pd.read_csv(OUTPUT_DIR / "tables" / "secondary_metric_intervals.csv")

display(Markdown("### Desempeño descriptivo por sitio y tamaño de ROI (16 filas)"))
display(descriptive_performance)

display(Markdown("### Contraste principal: AUC 12 − 116, por sitio (sin efecto combinado)"))
display(primary_12_vs_116)

display(Markdown("### Diagnóstico de precisión (error estándar bootstrap + IC bilateral; "
                  "sin margen ni cuantil unilateral)"))
display(precision_diagnostics)

display(Markdown("### Contrastes secundarios (100 filas: 4 sitios x 5 contrastes x 5 métricas)"))
display(secondary_pairwise.head(10))

display(Markdown("### Intervalos por métrica secundaria (64 filas)"))
display(secondary_intervals.head(10))

## 5. Figuras

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "figures" / "paired_roi_profiles.png")))
display(Image(filename=str(OUTPUT_DIR / "figures" / "primary_contrast_forest.png")))

## 6. Análisis de errores (12 vs. 116, por sujeto y repetición)

In [ ]:
error_summary = pd.read_csv(OUTPUT_DIR / "tables" / "error_analysis_summary.csv")
subject_error_profiles = pd.read_csv(OUTPUT_DIR / "data" / "subject_error_profiles.csv")
display(error_summary)
display(subject_error_profiles.head(10))

## 7. Narrativa obligatoria (sección 10.3, resolución D3)

El texto siguiente se **deriva** de `primary_12_vs_116.csv` (signos y
solapamiento de intervalos calculados, nunca escritos a mano) usando la
misma función que usa el script de análisis (`generate_d3_narrative`).

In [ ]:
narrative = rsa.generate_d3_narrative(primary_12_vs_116)
display(Markdown(f"> {narrative}"))

## 8. Limitaciones y reglas de comunicación (plan 5.6, secciones 1-2 y 15, y D1–D5)

- **Estado de preinscripción:** este análisis no es una preinscripción
  prospectiva ciega a los resultados. El plan 5.6 se cerró después de una
  revisión de factibilidad en la que ya eran visibles las diferencias medias
  y varianzas por sitio entre 12 y 116 ROIs. Los resultados se presentan
  como estimación con apoyo exploratorio, nunca como confirmación
  definitiva; una afirmación confirmatoria requeriría una cohorte o
  conjunto de datos externo no usado en estas decisiones.
- **D1 (métrica primaria):** media de los cinco AUC OOF por repetición.
  Balanced accuracy, F1-macro, sensibilidad y especificidad son secundarias;
  accuracy es solo auditoría.
- **D2 (margen):** no se definió un margen de no inferioridad. El análisis es
  de estimación pura: diferencias puntuales e intervalos bilaterales del 95%,
  sin un dictamen binario de "no inferioridad confirmada/rechazada".
- **D3 (agregación entre sitios):** cada sitio se presenta por separado. No
  se estima ni contrasta un efecto combinado. No se afirma ni se niega
  heterogeneidad estadística entre sitios; solo se describe la variación de
  las estimaciones puntuales.
- **D4 (bootstrap):** remuestreo pareado y estratificado por sujeto,
  condicionado a las predicciones, los entrenamientos y las cinco
  particiones de validación cruzada ya existentes. No captura la
  variabilidad de un reentrenamiento o una reparticipación completos.
- **D5 (estimando):** desempeño medio del *pipeline* de validación cruzada,
  no el de un ensamble de probabilidades promediadas ni el de un modelo
  final desplegado.
- La conclusión se expresa como *"estimamos la diferencia"*, nunca como
  *"confirmamos"* o *"refutamos"* que 12 ROIs alcanza a 116.

## 9. Checklist final y rutas de artefactos

In [ ]:
expected_outputs = [
    OUTPUT_DIR / "data" / "subject_scores.csv",
    OUTPUT_DIR / "data" / "metrics_by_repeat.csv",
    OUTPUT_DIR / "data" / "error_analysis_long.csv",
    OUTPUT_DIR / "data" / "subject_error_profiles.csv",
    OUTPUT_DIR / "tables" / "comparability_audit.csv",
    OUTPUT_DIR / "tables" / "descriptive_performance.csv",
    OUTPUT_DIR / "tables" / "primary_12_vs_116.csv",
    OUTPUT_DIR / "tables" / "precision_diagnostics.csv",
    OUTPUT_DIR / "tables" / "secondary_pairwise_comparisons.csv",
    OUTPUT_DIR / "tables" / "secondary_metric_intervals.csv",
    OUTPUT_DIR / "tables" / "error_analysis_summary.csv",
    OUTPUT_DIR / "figures" / "paired_roi_profiles.svg",
    OUTPUT_DIR / "figures" / "paired_roi_profiles.png",
    OUTPUT_DIR / "figures" / "primary_contrast_forest.svg",
    OUTPUT_DIR / "figures" / "primary_contrast_forest.png",
    OUTPUT_DIR / "analysis_manifest.json",
]
for p in expected_outputs:
    print(("[OK]  " if p.exists() else "[FALTA] "), p.relative_to(REPO_ROOT))

print()
import subprocess
try:
    status_lines = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_ROOT
    ).decode().splitlines()
    changed_outside_analysis = [l for l in status_lines if not l[3:].startswith("analysis/")]
    print("Cambios fuera de analysis/ (deben ser ninguno):", changed_outside_analysis or "(ninguno)")
except Exception as exc:
    print("No se pudo verificar git status:", exc)
print("Checklist de la sección 15: ver analysis/roi_comparison/README.md.")